# Ohlcv 1m Split-Normalized - Full Universe Audit Notebook `v0_1`

## Rol

Este notebook permite supervisar el **100% del universo auditado** de casos de split en `1m`.

No se apoya en PNGs incrustados como interfaz principal.
El inspector puede seleccionar cualquier caso del universo auditado y ver:

- metadatos del caso;
- estado de auditoría;
- cobertura antes y después del split;
- serie `raw`;
- serie `split_normalized`;
- y la trayectoria del `future_split_factor`.

## Pregunta auditada

- ¿todos los casos de split con cobertura suficiente pasan los invariantes semánticos?
- ¿y cómo se ve cada caso concreto cuando el inspector quiere comprobarlo uno a uno?

In [1]:
from pathlib import Path
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output
import sys

PROJECT_ROOT = Path(r'C:\TSIS_Data\01_TSIS_backtest_SmallCaps')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.price_views import apply_split_normalized_view, canonicalize_split_table

AUDIT_ROOT = PROJECT_ROOT / '01_foundations' / 'inspection_dossiers' / '1m_split_normalized' / 'evidence_assets' / 'full_universe_split_audit'
CASES = pd.read_parquet(AUDIT_ROOT / 'full_universe_split_event_cases.parquet')
META = pd.read_csv(AUDIT_ROOT / 'full_universe_split_event_audit_meta.csv')
STATUS = pd.read_csv(AUDIT_ROOT / 'full_universe_split_event_status_summary.csv')
MINUTE_ROOT = Path(r'D:\ohlcv_1m')
SPLITS_ROOT = Path(r'C:\TSIS_Data\data\additional\corporate_actions\splits')

display(Markdown('## 1. Resumen agregado del universo auditado'))
display(META)
display(STATUS)

## 1. Resumen agregado del universo auditado

,split_files_seen,non_empty_split_files_with_1m_ticker,total_event_cases,pass_cases,fail_cases,no_pre_cases,no_post_cases,no_1m_coverage_cases
0,4824,1876,3335,2280,0,164,151,740


,status,n_cases,pct_cases
0,NO_1M_COVERAGE,740,22.188906
1,NO_POST_COVERAGE,151,4.527736
2,NO_PRE_COVERAGE,164,4.917541
3,PASS,2280,68.365817


## 1. Cómo leer el resumen agregado

### Que muestra

- El conteo total de casos de split auditados con intersección real contra `1m`.
- Cuántos pasan, cuántos no tienen cobertura suficiente y cuántos fallan.

### Responde

- Si podemos afirmar que la semántica falla en algún caso cubierto.
- Si el universo pendiente se debe a errores o a límites de cobertura.

### Lectura técnica

- Un `FAIL = 0` significa que no encontramos ninguna violación de invariantes en los casos evaluables.
- `NO_PRE_COVERAGE` y `NO_POST_COVERAGE` no son fallos semánticos; son límites de lo que la historia disponible permite auditar antes o después del evento.
- `NO_1M_COVERAGE` significa que hay split en fuentes maestras, pero no historia `1m` suficiente en el dataset para ese ticker/evento.

### Consecuencia

- Este resumen separa con claridad error semántico real de ausencia de material empírico para auditar.

In [2]:
def month_path(ticker, year, month):
    return MINUTE_ROOT / f'ticker={ticker}' / f'year={year}' / f'month={month:02d}' / f'minute_aggs_{ticker}_{year}_{month:02d}.parquet'

def neighbor_months(event_date):
    d = pd.Timestamp(event_date).normalize().replace(day=1)
    prev = (d - pd.offsets.MonthBegin(1)).normalize()
    nxt = (d + pd.offsets.MonthBegin(1)).normalize()
    months = [(prev.year, prev.month), (d.year, d.month), (nxt.year, nxt.month)]
    out = []
    seen = set()
    for ym in months:
        if ym not in seen:
            seen.add(ym)
            out.append(ym)
    return out

def load_case_frame(case_row):
    ticker = case_row['ticker']
    event_date = pd.Timestamp(case_row['event_date']) if pd.notna(case_row['event_date']) else None
    if event_date is None:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
    frames = []
    for year, month in neighbor_months(event_date):
        p = month_path(ticker, year, month)
        if p.exists():
            df = pd.read_parquet(p).copy()
            if not df.empty:
                frames.append(df)
    if not frames:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
    raw = pd.concat(frames, ignore_index=True)
    raw['date'] = pd.to_datetime(raw['date'], errors='coerce')
    raw['ts_utc'] = pd.to_datetime(raw['ts_utc'], errors='coerce', utc=True)
    split_file = SPLITS_ROOT / f'ticker={ticker}' / f'splits_{ticker}.parquet'
    splits = pd.read_parquet(split_file) if split_file.exists() else pd.DataFrame()
    if '_empty' in splits.columns and len(splits) and bool(splits['_empty'].iloc[0]) is True:
        splits = pd.DataFrame()
    norm, _ = apply_split_normalized_view(raw, splits=splits, date_col='date', price_cols=('o','h','l','c','vw'))
    daily = norm.groupby('date', as_index=False).agg(
        open_raw=('o','first'), close_raw=('c','last'), high_raw=('h','max'), low_raw=('l','min'),
        open_norm=('o_split_normalized','first'), close_norm=('c_split_normalized','last'), high_norm=('h_split_normalized','max'), low_norm=('l_split_normalized','min'),
        future_split_factor=('future_split_factor','max'),
        minute_rows=('c','size')
    )
    return raw, norm, daily.sort_values('date').reset_index(drop=True)

def render_case(case_row):
    clear_output(wait=True)
    display(ui)
    display(Markdown('### Metadatos del caso'))
    meta_cols = [c for c in ['ticker','event_date','split_from','split_to','split_ratio','status','days_total','days_pre','days_post','rows_total','rows_pre','rows_post','months_loaded'] if c in case_row.index]
    display(pd.DataFrame([case_row[meta_cols]]))
    raw, norm, daily = load_case_frame(case_row)
    if daily.empty:
        display(Markdown('**Sin datos `1m` cargables para este caso.** Esto corresponde a `NO_1M_COVERAGE`.'))
        return
    event_date = pd.Timestamp(case_row['event_date'])
    fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True, constrained_layout=True)
    axes[0].plot(daily['date'], daily['close_raw'], marker='o', linewidth=1.6, color='#d95f02', label='close raw')
    axes[0].axvline(event_date, color='crimson', linestyle='--', linewidth=1.2, label='split date')
    axes[0].set_title('Serie daily agregada desde 1m | raw')
    axes[0].legend(loc='upper left', fontsize=8)
    axes[0].grid(True, alpha=0.25)
    axes[1].plot(daily['date'], daily['close_norm'], marker='o', linewidth=1.6, color='#1b9e77', label='close split_normalized')
    axes[1].axvline(event_date, color='crimson', linestyle='--', linewidth=1.2, label='split date')
    axes[1].set_title('Serie daily agregada desde 1m | split_normalized')
    axes[1].legend(loc='upper left', fontsize=8)
    axes[1].grid(True, alpha=0.25)
    axes[2].step(daily['date'], daily['future_split_factor'], where='mid', color='#386cb0', linewidth=1.8, label='future_split_factor')
    axes[2].axvline(event_date, color='crimson', linestyle='--', linewidth=1.2, label='split date')
    axes[2].axhline(1.0, color='black', linewidth=0.8, alpha=0.5)
    axes[2].set_title('Factor activo por día')
    axes[2].legend(loc='upper left', fontsize=8)
    axes[2].grid(True, alpha=0.25)
    plt.show()
    display(Markdown('### Lectura técnica automática'))
    status = str(case_row['status'])
    if status == 'PASS':
        display(Markdown('- **PASS**: este caso tiene cobertura antes y después del evento y no presenta ninguna violación de invariantes.'))
    elif status == 'NO_PRE_COVERAGE':
        display(Markdown('- **NO_PRE_COVERAGE**: el dataset no conserva suficiente historia previa para falsar la parte pre-evento, pero la parte disponible no enseña error semántico.'))
    elif status == 'NO_POST_COVERAGE':
        display(Markdown('- **NO_POST_COVERAGE**: el dataset no conserva suficiente historia posterior para falsar la parte post-evento, pero la parte disponible no enseña error semántico.'))
    elif status == 'NO_1M_COVERAGE':
        display(Markdown('- **NO_1M_COVERAGE**: existe split en fuentes maestras, pero no historia `1m` disponible para este evento.'))
    else:
        display(Markdown('- **FAIL**: el caso muestra violación de invariantes y debe revisarse.'))


In [3]:
status_options = ['ALL'] + sorted(CASES['status'].dropna().unique().tolist())
status_dd = widgets.Dropdown(options=status_options, value='ALL', description='status')
ticker_dd = widgets.Dropdown(description='ticker')
event_dd = widgets.Dropdown(description='evento')

def filter_cases():
    df = CASES.copy()
    if status_dd.value != 'ALL':
        df = df[df['status'] == status_dd.value]
    return df.sort_values(['ticker','event_date']).reset_index(drop=True)

def refresh_tickers(*args):
    df = filter_cases()
    tickers = sorted(df['ticker'].dropna().unique().tolist())
    ticker_dd.options = tickers
    ticker_dd.value = tickers[0] if tickers else None

def refresh_events(*args):
    df = filter_cases()
    if ticker_dd.value is not None:
        df = df[df['ticker'] == ticker_dd.value]
    labels = []
    for _, row in df.iterrows():
        label = f"{row['event_date']} | {row['split_from']}->{row['split_to']} | {row['status']}"
        labels.append((label, int(row.name)))
    event_dd.options = labels
    event_dd.value = labels[0][1] if labels else None

def on_change(*args):
    df = filter_cases()
    if ticker_dd.value is not None:
        df = df[df['ticker'] == ticker_dd.value]
    if event_dd.value is None or df.empty:
        clear_output(wait=True)
        display(ui)
        return
    case_row = df.loc[event_dd.value]
    render_case(case_row)

status_dd.observe(refresh_tickers, names='value')
status_dd.observe(lambda change: refresh_events(), names='value')
ticker_dd.observe(refresh_events, names='value')
ticker_dd.observe(on_change, names='value')
event_dd.observe(on_change, names='value')

ui = widgets.VBox([
    widgets.HTML('<h2>2. Selector del universo auditado</h2>'),
    widgets.HBox([status_dd, ticker_dd]),
    event_dd,
])

refresh_tickers()
refresh_events()
display(ui)
on_change()

### Metadatos del caso

,ticker,event_date,split_from,split_to,split_ratio,status,rows_total,rows_pre,rows_post,months_loaded
0,AAIC,2009-10-07,20.0,1.0,0.05,NO_1M_COVERAGE,0,0,0,


**Sin datos `1m` cargables para este caso.** Esto corresponde a `NO_1M_COVERAGE`.

## 2. Cómo usar el selector

### Qué hace

- `status` filtra por `PASS`, `NO_PRE_COVERAGE`, `NO_POST_COVERAGE` o `NO_1M_COVERAGE`.
- `ticker` acota al activo.
- `evento` selecciona la fecha concreta del split y muestra su detalle.

### Qué permite auditar

- Cualquier caso del universo cubierto por la auditoría.
- No solo los ejemplos bonitos del piloto.

### Por qué esto es importante

- La auditoría full-universe no debe quedarse en tablas agregadas; el inspector debe poder saltar desde el resumen a cualquier caso individual y ver la serie `raw`, la serie resuelta y el factor activo.

## 3. Veredicto final

- `FAIL = 0` en el universo con cobertura real evaluable.
- `PASS = 2280` casos con cobertura antes y después del evento.
- Los demás casos no son errores semánticos: son límites de cobertura (`NO_PRE_COVERAGE`, `NO_POST_COVERAGE`, `NO_1M_COVERAGE`).

La lectura institucional correcta es:

- **100% de los casos plenamente auditables pasan**.
- No afirmamos resolución empírica sobre eventos para los que el propio dataset no conserva historia suficiente.
